# Notebook 3 — Credit Risk KPIs & Decision Metrics

**Project:** Credit Risk & KPI Decision System  
**Goal:** Define business-facing KPIs from `clean_loans`, create consistent default definitions, and generate policy-ready KPI tables for decisioning.

---

## Notes on definitions
- **Observed outcomes set** (for realised default KPIs): `Fully Paid`, `Charged Off`, `Default`
- **Default flag**: 1 for `Charged Off`/`Default`, 0 for `Fully Paid`, NULL otherwise
- **Cashflow fields** are used only for descriptive KPIs (not for PD modelling features)

In [1]:
# Imports
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
DB_PATH = PROJECT_ROOT / "data" / "db" / "credit_risk.db"

assert DB_PATH.exists(), f"Database not found at: {DB_PATH}"
con = sqlite3.connect(DB_PATH)
con.execute("SELECT COUNT(*) FROM clean_loans;").fetchone()

(2260701,)

## 2) KPI base view (`loan_kpi_base`)
This view standardises the default flag and creates helper fields used across KPI queries.

**Helper fields**
- `is_default` (observed only)
- `issue_year`, `issue_month` derived from `issue_date` (`YYYY-MM-01`)
- `exposure` as `loan_amnt` (v1)


In [4]:
con.execute("DROP VIEW IF EXISTS loan_kpi_base;")

con.execute(
    '''
    CREATE VIEW loan_kpi_base AS
    SELECT
      *,
      CASE
        WHEN loan_status IN ('Charged Off', 'Default') THEN 1
        WHEN loan_status = 'Fully Paid' THEN 0
        ELSE NULL
      END AS is_default,
      CAST(substr(issue_date, 1, 4) AS INTEGER) AS issue_year,
      CAST(substr(issue_date, 6, 2) AS INTEGER) AS issue_month,
      CAST(loan_amnt AS REAL) AS exposure
    FROM clean_loans;
    '''
)
con.commit()

# Check mapping
pd.read_sql_query(
    "SELECT loan_status, is_default, COUNT(*) AS n FROM loan_kpi_base GROUP BY loan_status, is_default ORDER BY n DESC;",
    con
)

,loan_status,is_default,n
0,Fully Paid,0.0,1076751
1,Current,NaN,878317
2,Charged Off,1.0,268559
3,Late (31-120 days),NaN,21467
4,In Grace Period,NaN,8436
5,Late (16-30 days),NaN,4349
6,Does not meet the credit policy. Status:Fully ...,NaN,1988
7,Does not meet the credit policy. Status:Charge...,NaN,761
8,Default,1.0,40
9,None,NaN,33


## 3) Portfolio summary KPIs
High-level portfolio KPIs for loans with observed outcomes (`is_default IS NOT NULL`).


In [5]:
portfolio_summary = pd.read_sql_query(
    '''
    SELECT
      COUNT(*)                                            AS observed_loans,
      SUM(exposure)                                       AS total_exposure,
      AVG(exposure)                                       AS avg_loan_amount,
      ROUND(AVG(int_rate_pct), 3)                         AS avg_int_rate_pct,
      ROUND(AVG(term_months), 3)                          AS avg_term_months,
      SUM(is_default)                                     AS defaults,
      ROUND(AVG(is_default) * 100, 3)                     AS default_rate_pct,

      -- descriptive realised cashflow metrics (not used as PD features)
      SUM(total_rec_prncp)                                AS total_principal_repaid,
      SUM(total_rec_int)                                  AS total_interest_repaid,
      SUM(total_rec_late_fee)                             AS total_late_fees,
      SUM(recoveries)                                     AS total_recoveries
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL;
    ''',
    con
)
portfolio_summary

,observed_loans,total_exposure,avg_loan_amount,avg_int_rate_pct,avg_term_months,defaults,default_rate_pct,total_principal_repaid,total_interest_repaid,total_late_fees,total_recoveries
0,1345350,1.939991e+10,14419.969952,13.24,41.79,268599,19.965,1.638862e+10,3.227466e+09,2.131172e+06,3.248105e+08


In [6]:
con.execute("DROP VIEW IF EXISTS kpi_portfolio_summary;")
con.execute(
    '''
    CREATE VIEW kpi_portfolio_summary AS
    SELECT
      COUNT(*)                                            AS observed_loans,
      SUM(exposure)                                       AS total_exposure,
      AVG(exposure)                                       AS avg_loan_amount,
      AVG(int_rate_pct)                                   AS avg_int_rate_pct,
      AVG(term_months)                                    AS avg_term_months,
      SUM(is_default)                                     AS defaults,
      AVG(is_default)                                     AS default_rate,
      SUM(total_rec_prncp)                                AS total_principal_repaid,
      SUM(total_rec_int)                                  AS total_interest_repaid,
      SUM(total_rec_late_fee)                             AS total_late_fees,
      SUM(recoveries)                                     AS total_recoveries
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL;
    '''
)
con.commit()
"View created: kpi_portfolio_summary"

'View created: kpi_portfolio_summary'

## 4) KPI by grade

In [7]:
kpi_by_grade = pd.read_sql_query(
    '''
    SELECT
      grade,
      COUNT(*)                                 AS observed_loans,
      SUM(exposure)                            AS total_exposure,
      ROUND(AVG(exposure), 2)                  AS avg_loan_amount,
      ROUND(AVG(int_rate_pct), 3)              AS avg_int_rate_pct,
      SUM(is_default)                          AS defaults,
      ROUND(AVG(is_default) * 100, 3)          AS default_rate_pct,
      ROUND(SUM(recoveries) / NULLIF(SUM(exposure), 0) * 100, 3) AS recovery_rate_pct_of_exposure
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL
    GROUP BY grade
    ORDER BY grade;
    ''',
    con
)
kpi_by_grade

,grade,observed_loans,total_exposure,avg_loan_amount,avg_int_rate_pct,defaults,default_rate_pct,recovery_rate_pct_of_exposure
0,A,235095,3.266021e+09,13892.34,7.113,14206,6.043,0.353
1,B,392748,5.199050e+09,13237.62,10.679,52576,13.387,0.910
2,C,381694,5.415625e+09,14188.39,14.021,85657,22.441,1.737
3,D,200966,3.069207e+09,15272.27,17.722,61067,30.387,2.541
4,E,93656,1.650055e+09,17618.25,21.138,36041,38.482,3.501
5,F,32059,6.119320e+08,19087.68,24.935,14492,45.204,4.423
6,G,9132,1.880170e+08,20588.81,27.726,4560,49.934,4.827


In [8]:
con.execute("DROP VIEW IF EXISTS kpi_by_grade;")
con.execute(
    '''
    CREATE VIEW kpi_by_grade AS
    SELECT
      grade,
      COUNT(*)                                  AS observed_loans,
      SUM(exposure)                             AS total_exposure,
      AVG(exposure)                             AS avg_loan_amount,
      AVG(int_rate_pct)                         AS avg_int_rate_pct,
      SUM(is_default)                           AS defaults,
      AVG(is_default)                           AS default_rate,
      SUM(recoveries) / NULLIF(SUM(exposure), 0) AS recovery_rate_of_exposure
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL
    GROUP BY grade;
    '''
)
con.commit()
"View created: kpi_by_grade"

'View created: kpi_by_grade'

## 5) KPI by sub-grade

In [9]:
kpi_by_sub_grade = pd.read_sql_query(
    '''
    SELECT
      sub_grade,
      grade,
      COUNT(*)                        AS observed_loans,
      SUM(exposure)                   AS total_exposure,
      ROUND(AVG(int_rate_pct), 3)     AS avg_int_rate_pct,
      SUM(is_default)                 AS defaults,
      ROUND(AVG(is_default) * 100, 3) AS default_rate_pct
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL
    GROUP BY sub_grade, grade
    ORDER BY grade, sub_grade;
    ''',
    con
)
kpi_by_sub_grade.head(20)

,sub_grade,grade,observed_loans,total_exposure,avg_int_rate_pct,defaults,default_rate_pct
0,A1,A,43679,6.125042e+08,5.538,1409,3.226
1,A2,A,37178,4.974814e+08,6.522,1734,4.664
2,A3,A,37997,5.171304e+08,7.120,2094,5.511
3,A4,A,52236,7.339804e+08,7.513,3588,6.869
4,A5,A,64005,9.049244e+08,8.201,5381,8.407
5,B1,B,71154,9.405380e+08,8.905,7416,10.422
6,B2,B,74025,9.855396e+08,9.907,8410,11.361
7,B3,B,81828,1.103214e+09,10.749,10625,12.985
8,B4,B,83200,1.104888e+09,11.494,12337,14.828
9,B5,B,82541,1.064870e+09,12.010,13788,16.704


In [10]:
con.execute("DROP VIEW IF EXISTS kpi_by_sub_grade;")
con.execute(
    '''
    CREATE VIEW kpi_by_sub_grade AS
    SELECT
      sub_grade,
      grade,
      COUNT(*)          AS observed_loans,
      SUM(exposure)     AS total_exposure,
      AVG(int_rate_pct) AS avg_int_rate_pct,
      SUM(is_default)   AS defaults,
      AVG(is_default)   AS default_rate
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL
    GROUP BY sub_grade, grade;
    '''
)
con.commit()
"View created: kpi_by_sub_grade"

'View created: kpi_by_sub_grade'

## 6) KPI by vintage (issue year)
Vintage analysis shows how performance changes over time.


In [11]:
kpi_by_vintage = pd.read_sql_query(
    '''
    SELECT
      issue_year,
      COUNT(*)                        AS observed_loans,
      SUM(exposure)                   AS total_exposure,
      ROUND(AVG(int_rate_pct), 3)     AS avg_int_rate_pct,
      SUM(is_default)                 AS defaults,
      ROUND(AVG(is_default) * 100, 3) AS default_rate_pct
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL
    GROUP BY issue_year
    ORDER BY issue_year;
    ''',
    con
)
kpi_by_vintage.head(25)

,issue_year,observed_loans,total_exposure,avg_int_rate_pct,defaults,default_rate_pct
0,2007,251,2.219275e+06,10.315,45,17.928
1,2008,1562,1.439028e+07,11.156,247,15.813
2,2009,4716,4.643632e+07,12.190,594,12.595
3,2010,11536,1.221212e+08,11.749,1487,12.890
4,2011,21721,2.616838e+08,12.223,3297,15.179
5,2012,53367,7.184110e+08,13.638,8644,16.197
6,2013,134804,1.982613e+09,14.532,21024,15.596
7,2014,223103,3.253490e+09,13.656,41162,18.450
8,2015,375546,5.498601e+09,12.385,75804,20.185
9,2016,293105,4.240362e+09,13.087,68252,23.286


In [12]:
con.execute("DROP VIEW IF EXISTS kpi_by_vintage;")
con.execute(
    '''
    CREATE VIEW kpi_by_vintage AS
    SELECT
      issue_year,
      COUNT(*)          AS observed_loans,
      SUM(exposure)     AS total_exposure,
      AVG(int_rate_pct) AS avg_int_rate_pct,
      SUM(is_default)   AS defaults,
      AVG(is_default)   AS default_rate
    FROM loan_kpi_base
    WHERE is_default IS NOT NULL
    GROUP BY issue_year;
    '''
)
con.commit()
"View created: kpi_by_vintage"

'View created: kpi_by_vintage'

## 7) Simple approval-policy scaffolding (rule-based)
This is not the PD model yet — it’s a **KPI-driven decision rule** to demonstrate risk appetite.

Example: approve only grades up to a chosen maximum grade.
- Conservative: approve `A–B`
- Balanced: approve `A–C`
- Growth: approve `A–D`

In [13]:
policy = pd.DataFrame({
    "policy_name": ["conservative", "balanced", "growth"],
    "max_grade_inclusive": ["B", "C", "D"]
})
policy

,policy_name,max_grade_inclusive
0,conservative,B
1,balanced,C
2,growth,D


In [14]:
grade_order = pd.DataFrame({"grade": list("ABCDEFG"), "grade_rank": [1,2,3,4,5,6,7]})
grade_order.to_sql("grade_order", con, if_exists="replace", index=False)
policy.to_sql("approval_policy", con, if_exists="replace", index=False)

con.execute("CREATE INDEX IF NOT EXISTS idx_grade_order_grade ON grade_order(grade);")
con.execute("CREATE INDEX IF NOT EXISTS idx_policy_name ON approval_policy(policy_name);")
con.commit()

"Policy tables written to SQLite."


'Policy tables written to SQLite.'

### Evaluate each policy against historical outcomes
Estimate what would have happened if we had approved only loans within the policy grade band.


In [15]:
policy_eval = pd.read_sql_query(
    '''
    WITH base AS (
      SELECT
        l.*,
        go.grade_rank AS grade_rank
      FROM loan_kpi_base l
      JOIN grade_order go ON go.grade = l.grade
      WHERE l.is_default IS NOT NULL
    ),
    pol AS (
      SELECT
        p.policy_name,
        go.grade_rank AS max_grade_rank
      FROM approval_policy p
      JOIN grade_order go ON go.grade = p.max_grade_inclusive
    )
    SELECT
      p.policy_name,
      COUNT(*) FILTER (WHERE b.grade_rank <= p.max_grade_rank)                           AS approved_loans,
      ROUND(AVG(b.exposure) FILTER (WHERE b.grade_rank <= p.max_grade_rank), 2)          AS approved_avg_loan,
      ROUND(AVG(b.int_rate_pct) FILTER (WHERE b.grade_rank <= p.max_grade_rank), 3)      AS approved_avg_rate,
      ROUND(AVG(b.is_default) FILTER (WHERE b.grade_rank <= p.max_grade_rank) * 100, 3)  AS approved_default_rate_pct,
      SUM(b.exposure) FILTER (WHERE b.grade_rank <= p.max_grade_rank)                    AS approved_exposure
    FROM base b
    CROSS JOIN pol p
    GROUP BY p.policy_name
    ORDER BY approved_default_rate_pct;
    ''',
    con
)
policy_eval

,policy_name,approved_loans,approved_avg_loan,approved_avg_rate,approved_default_rate_pct,approved_exposure
0,conservative,627843,13482.78,9.344,10.637,8.465070e+09
1,balanced,1009537,13749.57,11.112,15.100,1.388070e+10
2,growth,1210503,14002.36,12.210,17.638,1.694990e+10


In [17]:
OUT_DIR = PROJECT_ROOT / "data" / "kpi"
OUT_DIR.mkdir(parents=True, exist_ok=True)

portfolio_summary.to_csv(OUT_DIR / "kpi_portfolio_summary.csv", index=False)
kpi_by_grade.to_csv(OUT_DIR / "kpi_by_grade.csv", index=False)
kpi_by_sub_grade.to_csv(OUT_DIR / "kpi_by_sub_grade.csv", index=False)
kpi_by_vintage.to_csv(OUT_DIR / "kpi_by_vintage.csv", index=False)
policy_eval.to_csv(OUT_DIR / "kpi_policy_evaluation.csv", index=False)

OUT_DIR

PosixPath('/home/jovyan/Home/notebooks/data/kpi')